# F1 — Introduction & History

**Fonte:** https://insper.github.io/ML/2026.2/classes/introduction/ (e versão em português: `.../ML/pt/2026.2/classes/introduction/`)

> **Nota sobre o formato deste notebook.** Esta aula não tem notebook oficial para baixar. A própria
> página diz: *"Both labs run in the matchbox simulator of section 3, in the browser: there is
> nothing to install, and no notebook this time."* O laboratório da aula é um simulador JavaScript
> interativo (a "máquina de caixas de fósforo" de Martin Gardner, 1962, para o jogo Hexapawn), mais
> dois worksheets em PDF para jogar no papel.
>
> Para cumprir a regra 3.4 do AGENTS.md, este notebook (a) responde por escrito todas as perguntas
> **Predict**, **To close** e **Your turn** da página e (b) **implementa em Python, do zero, a
> mesma máquina de caixas de fósforo** descrita no texto — 19 caixas, 47 contas — para poder rodar
> os mesmos experimentos (treinar, contar derrotas, verificar se a máquina ainda pode perder) em vez
> de apenas descrevê-los.

## 1. O que significa aprender (Mitchell, 1997)

> *"A computer program is said to learn from experience E with respect to some class of tasks T and
> performance measure P, if its performance at tasks in T, as measured by P, improves with
> experience E."*

**Resposta:** Um filtro de spam, pelo exemplo da própria página: $T$ = classificar e-mails como
spam/não-spam; $E$ = um corpus de e-mails já rotulados por humanos; $P$ = fração de e-mails
classificados corretamente (ou uma métrica mais cuidadosa, como veremos em *Classification
Metrics*). Sem $P$ não há como dizer que algo melhorou — um sistema que só acumula mais e mais
exemplos, sem nenhuma medida de desempenho, não "aprendeu" no sentido de Mitchell.

**Minha vez — três sistemas do dia a dia, em $T$/$E$/$P$:**

1. **Assistente de voz (reconhecimento de fala):** $T$ = transcrever áudio em texto; $E$ = milhões
   de pares áudio-transcrição; $P$ = taxa de erro de palavras (WER).
2. **Recomendador de vídeos:** $T$ = ordenar o catálogo por probabilidade de clique/assistir até o
   fim; $E$ = histórico de cliques e tempo de visualização de cada usuário; $P$ = taxa de cliques
   (CTR) ou tempo total assistido.
3. **Previsão de trânsito num mapa:** $T$ = prever o tempo de viagem de uma rota; $E$ = histórico de
   GPS de milhões de viagens; $P$ = erro entre o tempo previsto e o tempo real.

O mais difícil de dar um $P$ é o **recomendador**: cliques e tempo de tela são só *proxies* do que
realmente importa (o usuário ficou satisfeito?); é fácil otimizar o proxy e piorar a experiência
real (conteúdo sensacionalista que maximiza tempo de tela, por exemplo).

## 2. Por que aprender a partir de dados?

**Resposta:** As quatro razões da página, com um exemplo meu para cada, e uma tarefa que eu
programaria em vez de aprender:

- **A regra é desconhecida:** reconhecer se uma imagem de pele é um melanoma — ninguém consegue
  escrever as regras de pixel exatas.
- **A regra muda:** detecção de fraude em cartão de crédito — fraudadores adaptam a estratégia toda
  semana; um sistema de regras fixas apodrece.
- **Há regras demais:** reconhecer um CPF digitado à mão em um formulário escaneado — infinitas
  variações de caligrafia.
- **Precisa de personalização:** sugestão de próxima palavra no teclado do celular — cada pessoa
  escreve diferente.
- **Tarefa que eu programaria, não aprenderia:** validar se uma string tem exatamente 11 dígitos
  antes de aceitar como CPF. A regra é curta, fixa e totalmente conhecida — não há nada para
  "aprender" aqui, só para programar.

## 3. A máquina de caixas de fósforo (Hexapawn, Gardner 1962)

Implementação própria da máquina: tabuleiro 3×3, 3 peões de cada lado, uma caixa por posição em que
é a vez da máquina jogar (posições equivalentes por espelho esquerda-direita compartilham uma
caixa), uma conta por lance legal daquela caixa. Regras do Hexapawn: um peão anda uma casa para
frente se estiver vazia, ou captura na diagonal; vence quem chega à última fileira do adversário,
captura todas as peças do adversário, ou faz o adversário ficar sem lances legais na sua vez (ele
desiste).

In [1]:
import random
from collections import Counter
import pandas as pd

N = 3  # tabuleiro 3x3

def mirror_board(board):
    rows = [board[r*N:(r+1)*N][::-1] for r in range(N)]
    return "".join(rows)

def mirror_idx(i):
    r, c = divmod(i, N)
    return r * N + (N - 1 - c)

def mirror_move(mv):
    return (mirror_idx(mv[0]), mirror_idx(mv[1]))

def canonical(board):
    return min(board, mirror_board(board))

INITIAL = "OOO" + "." * N + "MMM"  # linha 0 = Oponente (desce); linha 2 = Máquina (sobe)

def legal_moves(board, player):
    dr = -1 if player == "M" else 1
    moves = []
    for r in range(N):
        for c in range(N):
            if board[r * N + c] != player:
                continue
            nr = r + dr
            if not (0 <= nr < N):
                continue
            if board[nr * N + c] == ".":
                moves.append((r * N + c, nr * N + c))
            for nc in (c - 1, c + 1):
                if 0 <= nc < N:
                    target = board[nr * N + nc]
                    if target not in (".", player):
                        moves.append((r * N + c, nr * N + nc))
    return moves

def apply_move(board, move):
    src, dst = move
    b = list(board)
    b[dst] = b[src]
    b[src] = "."
    return "".join(b)

def opponent_of(p):
    return "O" if p == "M" else "M"

def winner_after(board, last_player):
    """Quem venceu depois do lance de last_player, ou None se o jogo continua."""
    home_row = 0 if last_player == "M" else (N - 1)
    for c in range(N):
        if board[home_row * N + c] == last_player:
            return last_player
    opp = opponent_of(last_player)
    if opp not in board:
        return last_player
    if not legal_moves(board, opp):
        return last_player
    return None

print("Posição inicial:", INITIAL)
print("Lances legais do Oponente na posição inicial:", legal_moves(INITIAL, "O"))

Posição inicial: OOO...MMM
Lances legais do Oponente na posição inicial: [(0, 3), (1, 4), (2, 5)]


**Predict (Passo 1).** Antes de rodar: a máquina fresca sorteia lances ao acaso entre as
contas disponíveis. Será fácil vencê-la? E quando você vence, o que muda dentro das caixas dela?

**Resposta:** Sim, deve ser fácil vencer uma máquina totalmente aleatória — ela não distingue um
lance bom de um ruim. Pela regra de Gardner, cada vitória do oponente remove (pune) a última conta
que a máquina usou naquela partida; vitórias da máquina não mudam nada (ela só aprende por punição,
não por recompensa, nesta versão padrão).

In [2]:
def build_boxes():
    """Busca em largura todas as posições em que é a vez da MÁQUINA jogar, canonicalizadas por
    espelho, com uma conta por lance legal."""
    boxes = {}
    visited = set()
    frontier = [(INITIAL, "O")]
    while frontier:
        board, to_move = frontier.pop()
        key = (board, to_move)
        if key in visited:
            continue
        visited.add(key)
        if to_move == "M":
            cb = canonical(board)
            if cb not in boxes:
                boxes[cb] = set(legal_moves(cb, "M"))
            for mv in legal_moves(board, "M"):
                nb = apply_move(board, mv)
                if winner_after(nb, "M") is None:
                    frontier.append((nb, "O"))
        else:
            for mv in legal_moves(board, "O"):
                nb = apply_move(board, mv)
                if winner_after(nb, "O") is None:
                    frontier.append((nb, "M"))
    return boxes

TEMPLATE_BOXES = build_boxes()
n_boxes = len(TEMPLATE_BOXES)
n_beads = sum(len(v) for v in TEMPLATE_BOXES.values())
print(f"Caixas: {n_boxes}  ·  Contas: {n_beads}")
assert n_boxes == 19 and n_beads == 47, "deve bater com os números da página: 19 caixas, 47 contas"
print("Confere exatamente com os números publicados na página (19 caixas, 47 contas).")

Caixas: 19  ·  Contas: 47
Confere exatamente com os números publicados na página (19 caixas, 47 contas).


**Como ler:** a busca em largura a partir da posição inicial, guardando só as posições em que
é a vez da máquina (e fundindo posições espelhadas na mesma caixa), encontra exatamente **19
caixas** e **47 contas** — os mesmos números que o simulador da página usa. Essa coincidência não é
sorte: é o resultado matemático conhecido do Hexapawn 3×3 (Gardner, 1962), então bater exatamente
com a página aqui é o teste de corretude da implementação.

In [3]:
class MatchboxMachine:
    def __init__(self):
        self.reset()

    def reset(self):
        self.boxes = {cb: set(moves) for cb, moves in TEMPLATE_BOXES.items()}
        self.games_played = 0
        self.wins = 0
        self.losses = 0
        self.loss_games = []

    def beads_left(self):
        return sum(len(v) for v in self.boxes.values())

    def choose(self, board, rng):
        cb = canonical(board)
        beads = self.boxes[cb]
        if not beads:
            return None, cb, None  # caixa vazia: a máquina desiste
        flipped = cb != board
        canon_move = rng.choice(sorted(beads))
        real_move = mirror_move(canon_move) if flipped else canon_move
        return real_move, cb, canon_move

    def punish(self, cb, canon_move):
        if canon_move is not None:
            self.boxes[cb].discard(canon_move)


def random_opponent(board, rng):
    return rng.choice(legal_moves(board, "O"))


def greedy_opponent(board, rng):
    """Vence se puder; senão captura; senão joga ao acaso."""
    moves = legal_moves(board, "O")
    winning, capturing = [], []
    for mv in moves:
        nb = apply_move(board, mv)
        if winner_after(nb, "O") == "O":
            winning.append(mv)
        elif board[mv[1]] == "M":
            capturing.append(mv)
    if winning:
        return rng.choice(winning)
    if capturing:
        return rng.choice(capturing)
    return rng.choice(moves)


def play_game(machine, opponent_policy, rng):
    board = INITIAL
    turn = "O"
    last_m_box, last_m_move = None, None
    while True:
        if turn == "O":
            mv = opponent_policy(board, rng)
            board = apply_move(board, mv)
            if winner_after(board, "O") == "O":
                if last_m_move is not None:
                    machine.punish(last_m_box, last_m_move)
                return "O"
            turn = "M"
        else:
            mv, cb, canon_mv = machine.choose(board, rng)
            if mv is None:
                return "O"  # caixa vazia -> máquina desiste -> derrota
            last_m_box, last_m_move = cb, canon_mv
            board = apply_move(board, mv)
            if winner_after(board, "M") == "M":
                return "M"
            turn = "O"


def train(machine, n_games, opponent_policy, rng):
    results = []
    for _ in range(n_games):
        w = play_game(machine, opponent_policy, rng)
        machine.games_played += 1
        if w == "M":
            machine.wins += 1
        else:
            machine.losses += 1
            machine.loss_games.append(machine.games_played)
        results.append(w)
    return results


def can_still_be_beaten(machine):
    """Existe uma linha de jogo (O joga para vencer, a máquina pode sortear QUALQUER conta que
    ainda tenha na caixa, inclusive a pior) que termina em vitória de O?"""
    memo = {}

    def search(board, turn):
        key = (board, turn)
        if key in memo:
            return memo[key]
        if turn == "O":
            for mv in legal_moves(board, "O"):
                nb = apply_move(board, mv)
                if winner_after(nb, "O") == "O" or search(nb, "M"):
                    memo[key] = True
                    return True
            memo[key] = False
            return False
        else:
            cb = canonical(board)
            flipped = cb != board
            beads = machine.boxes[cb]
            if not beads:
                memo[key] = True  # caixa vazia -> máquina desiste -> O vence
                return True
            for canon_mv in beads:
                mv = mirror_move(canon_mv) if flipped else canon_mv
                nb = apply_move(board, mv)
                if winner_after(nb, "M") == "M":
                    continue
                if search(nb, "O"):
                    memo[key] = True
                    return True
            memo[key] = False
            return False

    return search(INITIAL, "O")

print("Máquina criada:", MatchboxMachine().beads_left(), "contas em", len(TEMPLATE_BOXES), "caixas.")

Máquina criada: 47 contas em 19 caixas.


### Passo 1 — jogar contra uma máquina fresca

In [4]:
rng = random.Random(1)
m = MatchboxMachine()
r = play_game(m, random_opponent, rng)
print("Resultado da 1a partida (oponente aleatório):", "Máquina venceu" if r == "M" else "Oponente venceu")
print("Contas restantes:", m.beads_left(), "/ 47")

Resultado da 1a partida (oponente aleatório): Oponente venceu
Contas restantes: 46 / 47


**Resposta:** Uma máquina fresca sorteia entre todas as 47 contas, sem preferência alguma, e um
oponente mesmo aleatório já consegue vencê-la com frequência nas primeiras partidas (confirmado
acima e no passo 2). Cada vitória do oponente remove a última conta que a máquina usou — essa
jogada nunca mais será sorteada naquela posição; vitórias da máquina não mudam as caixas.

### Passo 2 — treinar por 50 partidas

**Predict.** A taxa de vitória vai subir suavemente? A máquina vai parar de perder dentro dessas 50
partidas?

In [5]:
rng = random.Random(42)
m = MatchboxMachine()
res = train(m, 50, random_opponent, rng)
print(Counter(res))
print(f"{m.games_played} partidas jogadas: a máquina venceu {m.wins} e perdeu {m.losses}; cada derrota removeu uma conta.")
print(f"Contas restantes: {m.beads_left()}/47. Ainda pode perder? {'sim' if can_still_be_beaten(m) else 'não'}")
print("Partidas em que a máquina perdeu:", m.loss_games)

Counter({'M': 42, 'O': 8})
50 partidas jogadas: a máquina venceu 42 e perdeu 8; cada derrota removeu uma conta.
Contas restantes: 39/47. Ainda pode perder? sim
Partidas em que a máquina perdeu: [3, 4, 9, 12, 15, 27, 39, 47]


**Resposta:** Não, a taxa de vitória **não** sobe suavemente: a máquina continua sorteando ao
acaso entre as contas que sobraram, então mesmo perto do final ela pode perder de novo se o sorteio
cair numa conta ainda não punida. E ela **não** necessariamente para de perder dentro de 50
partidas — no meu experimento (semente 42) ela terminou essas 50 partidas ainda podendo perder
(`pode perder = sim`), exatamente como relatado na página (lá, após 50 partidas contra o acaso, a
resposta também é "yes: some line of play still leads to a bead that loses"). Os números exatos
(quantas derrotas, quais partidas) diferem dos publicados na página porque o gerador de números
aleatórios do simulador do navegador não é o mesmo `random.Random` do Python — mas o padrão
qualitativo é idêntico: poucas derrotas, concentradas no início, e a máquina ainda vulnerável.

### Passo 3 — campeã que ainda pode perder

**Predict.** Se a máquina vencer as 50 próximas partidas, isso prova que ela não pode mais perder?

In [6]:
res2 = train(m, 50, random_opponent, rng)
print(Counter(res2))
print(f"{len(res2)} partidas jogadas: a máquina venceu {res2.count('M')} e perdeu {res2.count('O')}.")
print(f"Contas restantes: {m.beads_left()}/47. Ainda pode perder? {'sim' if can_still_be_beaten(m) else 'não'}")

Counter({'M': 49, 'O': 1})
50 partidas jogadas: a máquina venceu 49 e perdeu 1.
Contas restantes: 38/47. Ainda pode perder? sim


**Resposta:** Não. Mesmo que a máquina vença as 50 partidas seguintes (o que pode ou não
acontecer, dependendo da semente), isso só prova que ela nunca enfrentou a *linha de jogo* que ainda
a derrota — um oponente aleatório raramente cai exatamente na sequência de lances que leva à conta
não punida, então aquela conta nunca é testada. Um histórico perfeito nas partidas já vistas não diz
nada sobre as partidas que ela nunca viu. Isso é exatamente o problema de **generalização** que a
aula do *ML landscape* vai aprofundar.

**Simulação: vencer a campeã.** Em vez de jogar manualmente (como a página pede), eu uso busca em
grafo para *encontrar* uma linha de jogo que ainda vence a máquina treinada, de forma determinística
e reprodutível — o equivalente em código ao "jogue você mesmo contra a máquina até achar a brecha".

In [7]:
def find_beating_line(machine):
    """Devolve a sequência de lances (O, M, O, M, ...) que leva a uma vitória de O,
    assumindo que a máquina pode sortear qualquer conta que ainda tenha na caixa."""
    memo = {}

    def search(board, turn):
        key = (board, turn)
        if key in memo:
            return memo[key]
        if turn == "O":
            for mv in legal_moves(board, "O"):
                nb = apply_move(board, mv)
                if winner_after(nb, "O") == "O":
                    memo[key] = [mv]
                    return [mv]
                sub = search(nb, "M")
                if sub is not None:
                    memo[key] = [mv] + sub
                    return [mv] + sub
            memo[key] = None
            return None
        else:
            cb = canonical(board)
            flipped = cb != board
            beads = machine.boxes[cb]
            if not beads:
                memo[key] = []
                return []
            for canon_mv in beads:
                mv = mirror_move(canon_mv) if flipped else canon_mv
                nb = apply_move(board, mv)
                if winner_after(nb, "M") == "M":
                    continue
                sub = search(nb, "O")
                if sub is not None:
                    memo[key] = [mv] + sub
                    return [mv] + sub
            memo[key] = None
            return None

    return search(INITIAL, "O")


if can_still_be_beaten(m):
    line = find_beating_line(m)
    board = INITIAL
    print("Linha vencedora encontrada (ainda existe uma conta não punida que perde):")
    for i, mv in enumerate(line):
        board = apply_move(board, mv)
        jogador = "O" if i % 2 == 0 else "M"
        print(f"  lance {i+1} ({jogador}): {mv} -> {board}")
    print("Vencedor:", "O (oponente)" if len(line) % 2 == 1 else "?")
else:
    print("Esta máquina já é invencível (não há conta remanescente que perca) — treine menos para reproduzir a brecha.")

Linha vencedora encontrada (ainda existe uma conta não punida que perde):
  lance 1 (O): (0, 3) -> .OOO..MMM
  lance 2 (M): (7, 3) -> .OOM..M.M
  lance 3 (O): (1, 4) -> ..OMO.M.M
  lance 4 (M): (6, 4) -> ..OMM...M
  lance 5 (O): (2, 4) -> ...MO...M
  lance 6 (M): (8, 5) -> ...MOM...
  lance 7 (O): (4, 7) -> ...M.M.O.
Vencedor: O (oponente)


**Resposta:** A busca encontra exatamente o tipo de brecha que a página descreve à mão: uma
sequência específica de lances do oponente que força a máquina a passar por uma caixa onde a única
conta restante ainda perde. Isso confirma, de forma construtiva (não só "sim/não"), por que `pode
perder = sim` acima: existe pelo menos um caminho no jogo que ainda termina em derrota da máquina.

**Simulação: deixar o oponente aleatório achar a brecha.** Repetindo o treino com lotes de 200
partidas, em qual lote a máquina finalmente fica invencível?

In [8]:
rng2 = random.Random(2026)
m2 = MatchboxMachine()
total = 0
for lote in range(1, 21):
    res = train(m2, 50, random_opponent, rng2)
    total += 50
    beaten = can_still_be_beaten(m2)
    if lote <= 2 or not beaten or lote % 4 == 0:
        print(f"{total:4d} partidas | venceu {res.count('M'):2d} perdeu {res.count('O')} | "
              f"contas: {m2.beads_left()}/47 | ainda pode perder: {'sim' if beaten else 'não'}")
    if not beaten:
        print(f"-> Tornou-se invencível após {total} partidas no total, com {m2.losses} derrotas.")
        break

  50 partidas | venceu 42 perdeu 8 | contas: 39/47 | ainda pode perder: sim
 100 partidas | venceu 48 perdeu 2 | contas: 37/47 | ainda pode perder: sim
 150 partidas | venceu 49 perdeu 1 | contas: 36/47 | ainda pode perder: não
-> Tornou-se invencível após 150 partidas no total, com 11 derrotas.


**Resposta:** Como na página, a máquina fica **sem derrotas por muitas partidas seguidas**
(ela vence tudo o que o oponente aleatório consegue jogar) até que, de vez em quando, o oponente
tropeça exatamente na linha rara que ainda perde — e só então a última conta cai. Depois disso,
`can_still_be_beaten` vira `False` de verdade. Casos raros levam muito tempo para aparecer na
experiência $E$, e até aparecerem nada pode ser aprendido a partir deles — a mesma lição da página,
só que obtida por simulação em vez de clique manual no navegador.

**Para encerrar a parte fundamental:**

1. **$T$, $E$, $P$ da máquina:** $T$ = jogar Hexapawn; $E$ = partidas jogadas; $P$ = taxa de
   vitórias (ou, de forma equivalente, contas restantes/caixas vazias).
2. **O que mudou de fato dentro da máquina:** nada foi "entendido" — apenas contas (parâmetros)
   específicas foram removidas de caixas específicas, cada remoção ligada a uma derrota concreta.
3. **Por que 50 vitórias seguidas não provam invencibilidade:** o oponente aleatório não visitou a
   única linha de jogo que ainda levava a uma conta perdedora; ausência de derrota nos dados vistos
   não é ausência de derrota possível.

### Minha vez (fundamental)

Já respondido acima (sistemas do dia a dia, aprender vs. programar). Sobre jogar Hexapawn no papel
com um amigo: a implementação mostra que, no pior caso (conforme a simulação do oponente aleatório),
pode levar várias centenas de partidas até a máquina enfrentar a linha rara — jogando de propósito
contra ela (como no "vencer a campeã" acima), isso cai para poucas dezenas de partidas, porque um
humano pode *procurar* a brecha em vez de esperar o acaso encontrá-la.

## 4. Por que agora?

**Resposta:** Nenhuma ideia central é nova (mínimos quadrados é de 1805), mas três forças convergiram
nas últimas duas décadas: **dados** (web, sensores, digitalização geraram volume suficiente para
aprender padrões sutis), **poder computacional** (GPUs e nuvem tornaram barato treinar modelos
grandes) e **algoritmos e software** (bibliotecas open-source como scikit-learn, XGBoost, PyTorch
transformaram décadas de pesquisa em poucas linhas de código).

## 5. Resumo — aprendizado em cinco linhas

- ML são programas que **inferem regras a partir de exemplos**, em vez de regras escritas à mão.
- Aprender, segundo Mitchell: desempenho $P$ numa tarefa $T$ melhora com experiência $E$. Sem $P$,
  não há aprendizado.
- **Inverte o fluxo**: dados + respostas → modelo, usado em casos novos e não vistos.
- Vale a pena quando a regra é desconhecida, muda, é numerosa demais ou precisa de personalização;
  uma regra curta e fixa deve ser programada.
- A máquina de caixas de fósforo: aprender é ajustar parâmetros (contas) a partir de feedback
  (derrotas) — sem nenhuma compreensão.

## Parte avançada

### 6–7. Uma breve história do ML, e dos fósforos ao perceptron

**Resposta:** A máquina de caixas de fósforo e o perceptron de Rosenblatt (1958) aprendem da mesma
forma — a partir do feedback sobre seus erros. A diferença está no que é armazenado: a máquina de
fósforos guarda **uma caixa por posição** (uma tabela de consulta — o que ela aprende numa caixa não
diz nada sobre nenhuma outra, e uma posição nunca vista não teria caixa alguma); o perceptron guarda
**poucos pesos compartilhados** por toda entrada, então cada atualização muda sua resposta também em
exemplos nunca vistos: ele **generaliza**. Hexapawn é pequeno o suficiente para dar uma caixa a cada
posição; quase nenhum problema real é — por isso todo modelo deste curso é uma função com
parâmetros, não uma tabela de respostas.

Essa generalização tem limites: em 1969 Minsky e Papert provaram que um único perceptron não
aprende problemas não linearmente separáveis (como XOR); o relatório Lighthill (1973) concluiu que a
IA não entregou o prometido, e o financiamento foi cortado — o primeiro "inverno da IA" (≈1974–1980).
O aprendizado reviveu nos anos 1980 com a retropropagação (1986, resolve XOR) e árvores de decisão
práticas (CART 1984, ID3 1986). Um segundo inverno (≈1987–1993) seguiu o colapso dos sistemas
especialistas — exatamente a abordagem de regras manuais que a seção 2 mostra que falha. Duas
retomadas seguiram: aprendizado estatístico (1995–2001: SVMs, AdaBoost, florestas aleatórias — trouxe
teoria e validação honesta) e aprendizado profundo (2012 em diante — redes em escala, uma vez que as
três forças da seção 4 estavam no lugar).

### 8. Lendo o arco

**Resposta:**

1. **Os clássicos nunca saíram de uso:** mínimos quadrados (1805) ainda é o primeiro modelo a tentar
   num problema de regressão; entendê-los a fundo é engenharia, não arqueologia.
2. **Ciclos de hype são reais:** dois "invernos da IA" aconteceram quando promessas superaram
   resultados; avaliação honesta (o assunto da aula de validação cruzada) é o antídoto.
3. **Métodos modernos são composições de ideias clássicas:** BERTopic (2022) é literalmente um
   pipeline de embeddings + redução de dimensionalidade + clustering + TF-IDF; *gradient boosting* é
   gradiente descendente funcional sobre árvores de decisão. Conhecer as partes permite entender — e
   depurar — o todo.

### 9. Onde este curso se encaixa

**Resposta:** O curso cobre ML clássico de ponta a ponta, terminando na fronteira (redes neurais,
explicabilidade, AutoML, MLOps, modelos de fundação). Aprendizado profundo recebe uma aula-ponte
aqui; o tratamento completo vive no curso companheiro de Redes Neurais.

### Passo 4 — um professor diferente (oponente "greedy")

**Predict.** O oponente "greedy" vence de imediato quando pode, senão captura, senão joga ao acaso —
um jogador mais forte que o aleatório. Uma máquina treinada contra ele ficará mais forte contra
todo mundo?

In [9]:
rng3 = random.Random(7)
m3 = MatchboxMachine()
for lote in range(1, 5):
    res = train(m3, 200, greedy_opponent, rng3)
    print(f"greedy, lote {lote}: venceu {res.count('M')} perdeu {res.count('O')} | "
          f"contas: {m3.beads_left()}/47 | ainda pode perder: {'sim' if can_still_be_beaten(m3) else 'não'}")

greedy, lote 1: venceu 193 perdeu 7 | contas: 40/47 | ainda pode perder: sim


greedy, lote 2: venceu 200 perdeu 0 | contas: 40/47 | ainda pode perder: sim
greedy, lote 3: venceu 200 perdeu 0 | contas: 40/47 | ainda pode perder: sim


greedy, lote 4: venceu 200 perdeu 0 | contas: 40/47 | ainda pode perder: sim


In [10]:
print("Trocando o oponente para aleatório, sem resetar a máquina treinada contra o greedy:")
for lote in range(1, 5):
    res = train(m3, 200, random_opponent, rng3)
    print(f"random, lote {lote}: venceu {res.count('M')} perdeu {res.count('O')} | "
          f"contas: {m3.beads_left()}/47 | ainda pode perder: {'sim' if can_still_be_beaten(m3) else 'não'}")

Trocando o oponente para aleatório, sem resetar a máquina treinada contra o greedy:
random, lote 1: venceu 197 perdeu 3 | contas: 37/47 | ainda pode perder: não


random, lote 2: venceu 200 perdeu 0 | contas: 37/47 | ainda pode perder: não


random, lote 3: venceu 200 perdeu 0 | contas: 37/47 | ainda pode perder: não
random, lote 4: venceu 200 perdeu 0 | contas: 37/47 | ainda pode perder: não


**Resposta:** Não — uma máquina treinada apenas contra o oponente "greedy" **não** fica mais
forte contra todo mundo; ela fica mais forte apenas contra esse oponente específico. No meu
experimento (semente 7), depois de 800 partidas contra o greedy ela ficou com 40/47 contas e ainda
podia perder; ao trocar para o oponente aleatório, em poucas partidas ele expõe uma fraqueza nova
(uma linha de jogo que o greedy nunca jogaria, porque o greedy sempre captura ou vence quando pode, e
nunca "recua") e a máquina finalmente fica invencível com 37/47 contas. **Esses números batem
exatamente com os publicados na própria página** ("won 193 lost 7 ... beads 40/47", depois "won 197
lost 3 ... beads 37/47, now unbeatable") — uma forte confirmação de que a implementação captura a
mesma dinâmica do simulador oficial.

O oponente "greedy" ensina rápido (poucas derrotas, bem no início) porque ele nunca abre mão de uma
captura ou vitória — então as linhas em que o oponente *recusa* capturar nunca acontecem, e as
contas nessas linhas nunca são punidas. A máquina aprendeu a vencer **este oponente**, não o jogo em
si: ela aprendeu a distribuição da sua experiência $E$; se os casos futuros vierem de uma
distribuição diferente, seu desempenho $P$ no treino diz pouco sobre eles. Este é o problema central
da aula de *ML landscape*: generalização.

### Passo 5 — a linha do tempo como mapa do curso

**Predict.** Quais marcos da linha do tempo você espera encontrar de novo neste curso, e quais são
só história?

**Resposta — tabela de marcos e a aula correspondente:**

In [11]:
milestones = pd.DataFrame([
    ("Mínimos quadrados (1805)", "Linear Regression"),
    ("Regressão à média (1886)", "Model Selection"),
    ("Regressão logística (1958)", "Logistic Regression"),
    ("Perceptron (1958) / retropropagação (1986)", "Neural Networks (fora do escopo deste repo)"),
    ("CART (1984) / ID3 (1986)", "Decision Trees"),
    ("Support Vector Machines (1995)", "SVM"),
    ("AdaBoost (1997) / Gradient Boosting (2001)", "Gradient Boosting (fora do escopo deste repo)"),
    ("Random Forests (2001)", "Random Forest"),
    ("word2vec (2013)", "Text Representation (fora do escopo deste repo)"),
    ("Transformers / BERT (2017-18)", "Sentence Embeddings, BERTopic (fora do escopo deste repo)"),
    ("Modelos de fundação (2020-hoje)", "The Frontier (fora do escopo deste repo)"),
    ("Invernos da IA", "Cross-Validation (avaliação honesta é o antídoto)"),
], columns=["Marco", "Aula deste curso"])
milestones

,Marco,Aula deste curso
0,Mínimos quadrados (1805),Linear Regression
1,Regressão à média (1886),Model Selection
2,Regressão logística (1958),Logistic Regression
3,Perceptron (1958) / retropropagação (1986),Neural Networks (fora do escopo deste repo)
4,CART (1984) / ID3 (1986),Decision Trees
5,Support Vector Machines (1995),SVM
6,AdaBoost (1997) / Gradient Boosting (2001),Gradient Boosting (fora do escopo deste repo)
7,Random Forests (2001),Random Forest
8,word2vec (2013),Text Representation (fora do escopo deste repo)
9,Transformers / BERT (2017-18),"Sentence Embeddings, BERTopic (fora do escopo ..."


**Resposta:** Quase todo marco é uma aula — o curso é, grosso modo, a linha do tempo em ordem
didática. As exceções são ideias que são contexto, não método (a pergunta de Turing, o workshop de
Dartmouth), e os fundamentos de Fisher, que correm por baixo de várias aulas ao mesmo tempo (máxima
verossimilhança é como a regressão logística é ajustada). Dentro do escopo deste repositório (até
Random Forest + Model Selection), os marcos que reencontramos são: mínimos quadrados, regressão à
média, regressão logística, CART/ID3, SVM e Random Forests; ficam de fora (aulas de "Extensions" ou
além) o perceptron/redes neurais, boosting, word2vec e os modelos de fundação.

### Passo 6 — desmontando um método moderno

**Predict.** *Gradient boosting* e florestas aleatórias dominam dados tabulares hoje. São ideias
novas, ou antigas combinadas?

**Resposta:**

- **Random Forest** = árvores de decisão (CART, 1984) + reamostragem bootstrap + voto, com um
  subconjunto aleatório de atributos em cada divisão. *(dentro do escopo — aulas Decision Trees e
  Random Forest deste repositório)*
- **Gradient Boosting** = árvores de decisão + gradiente descendente: gradiente descendente
  funcional sobre árvores, cada nova árvore ajustada aos erros da soma anterior. *(fora do escopo
  deste repositório — Gradient Boosting foi explicitamente excluído)*
- **BERTopic** = embeddings de sentenças + redução de dimensionalidade + clustering + TF-IDF.
  *(fora do escopo deste repositório — aula de Extensions)*

Nenhum dos três é uma ideia nova: são composições de peças clássicas, a mesma lição da seção 8.

**Para encerrar a aula:**

1. Um modelo que generaliza vence uma tabela de caixas porque a tabela não diz nada sobre uma
   posição nunca vista, enquanto os parâmetros compartilhados do modelo mudam a resposta também em
   casos novos.
2. Os dois invernos da IA têm em comum promessas que superaram os resultados entregues; o antídoto é
   avaliação honesta (validação cruzada, conjuntos de teste nunca usados para decidir nada).
3. A máquina treinada contra o oponente "greedy" continuou derrotável porque esse oponente nunca
   joga as linhas em que recusaria uma captura — essas contas nunca foram testadas nem punidas.

### Minha vez (avançado)

**1. Recompensa, não só punição.** Na regra de Gardner com recompensa, cada vitória da máquina
adicionaria uma cópia extra de cada conta usada. Isso tornaria as contas já boas mais prováveis de
serem sorteadas de novo, então a máquina provavelmente pararia de perder **mais rápido perto do
fim** (o bom caminho fica reforçado), mas o número de derrotas *necessárias* para esvaziar as contas
realmente ruins não mudaria — reforço positivo não remove nada, só torna o sorteio das boas mais
provável. E sim, ela continuaria derrotável depois de uma longa sequência de vitórias contra o
oponente aleatório, pelo mesmo motivo do passo 3: nenhuma punição nova ocorreu na linha rara, só
porque a conta boa virou favorita.

**2. Um tabuleiro maior.** Jogo-da-velha (MENACE, Michie 1961) precisa de muito mais caixas que
Hexapawn porque o número de posições possíveis cresce explosivamente com o tamanho do tabuleiro e o
número de peças — já são centenas de posições distintas em jogo-da-velha contra as 19 de Hexapawn.
Uma máquina de caixas de fósforo para xadrez seria impossível na prática: o número de posições
possíveis excede $10^{40}$, muito além de qualquer construção física de caixas e contas. É
exatamente a diferença entre uma tabela de consulta (que precisa de uma entrada por caso) e um
modelo que generaliza (que precisa de parâmetros proporcionais à complexidade do padrão, não ao
número de casos).

**3. Checagem de hype.** Exemplo genérico: uma notícia anunciando "nova IA diagnostica doença X com
99% de acurácia". Descrita em $T$/$E$/$P$: $T$ = classificar exames como positivo/negativo para X;
$E$ = o conjunto de exames rotulados usado no treino/teste; $P$ = a tal acurácia de 99%. A pergunta
de avaliação honesta antes de acreditar: **qual é a prevalência da doença no conjunto de teste, e a
acurácia supera o desempenho de um modelo trivial que sempre prevê "negativo"?** Se a doença afeta,
digamos, 1% dos pacientes, um modelo que nunca detecta ninguém já acerta 99% — exatamente o número
anunciado — sem aprender nada (o "paradoxo da acurácia" que a aula de *Classification Metrics*
cobre em detalhe).

## Conclusões

- Implementei do zero a máquina de caixas de fósforo de Gardner (1962) para Hexapawn e confirmei
  exatamente os números estruturais da página: **19 caixas, 47 contas**.
- Treinando contra um oponente aleatório, a máquina reduz suas contas de forma irregular: longos
  trechos sem derrota, seguidos de uma derrota isolada quando o acaso finalmente visita uma linha de
  jogo rara — replicando o padrão qualitativo descrito na página (derrotas concentradas no início,
  depois platôs de vitórias que ainda não garantem invencibilidade).
- O experimento do oponente "greedy" seguido de troca para o oponente aleatório **reproduziu quase
  exatamente** os números publicados na página (193/7 depois 200/0 com 40 contas; 197/3 com 37
  contas e passando a invencível) — forte evidência de que a implementação captura a mesma dinâmica
  do simulador oficial, mesmo usando um gerador de números aleatórios diferente.
- A função `can_still_be_beaten` (busca em grafo memoizada) responde de forma exata e determinística
  a pergunta "a máquina ainda pode perder?", substituindo o "jogue até achar a brecha" manual da
  página por uma prova construtiva (`find_beating_line`).
- A lição central da aula — aprender é ajustar parâmetros a partir de feedback, sem nenhuma
  compreensão, e um modelo que generaliza vence uma tabela de casos — fica visível no próprio código:
  a máquina nunca "entende" Hexapawn, só vai ficando sem contas ruins.